In [ ]:
import struct
import sys
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
plt.rcParams['figure.figsize']=(6,6)
plt.rcParams['font.weight']='bold'
plt.rcParams['axes.labelweight']='bold'
plt.rcParams['lines.linewidth']=2
plt.rcParams['lines.markeredgewidth']=2
%matplotlib inline
%config InlineBackend.figure_format = "retina"

thincurr_python_path = '/home/clair/repos/install_release'
if thincurr_python_path is not None:
    sys.path.append(os.path.join(thincurr_python_path,'python'))
from OpenFUSIONToolkit import OFT_env
from OpenFUSIONToolkit.ThinCurr import ThinCurr
from OpenFUSIONToolkit.ThinCurr.meshing import build_torus_bnorm_grid, ThinCurr_periodic_toroid
from OpenFUSIONToolkit.ThinCurr.sensor import Mirnov, save_sensors
from OpenFUSIONToolkit.ThinCurr.valen import VALENSystem 

def create_circular_bnorm(filename,R0,Z0,a,n,m,npts=200):
    theta_vals = np.linspace(0.0,2*np.pi,npts,endpoint=False)
    with open(filename,'w+') as fid:
        fid.write('{0} {1}\n'.format(npts,n))
        for theta in theta_vals:
            fid.write('{0} {1} {2} {3}\n'.format(
                R0+a*np.cos(theta),
                Z0+a*np.sin(theta),
                np.cos(m*theta),
                np.sin(m*theta)
            ))
# Create n=2, m=3 mode
create_circular_bnorm('tCurr_mode.dat',1.0,0.0,0.4,2,3)


ntheta = 40
nphi = 80
r_grid, bnorm, nfp = build_torus_bnorm_grid('tCurr_mode.dat',ntheta,nphi,resample_type='theta',use_spline=False)
plasma_mode = ThinCurr_periodic_toroid(r_grid,nfp,ntheta,nphi)

plasma_mode.write_to_file('thincurr_mode.h5')


myOFT = OFT_env(nthreads=4)
tw_mode = ThinCurr(myOFT)
tw_mode.setup_model(mesh_file='thincurr_mode.h5')
tw_mode.setup_io(basepath='plasma/')

tw_torus = ThinCurr(myOFT)
tw_torus.setup_model(mesh_file='thincurr_ex-torus.h5',xml_filename='valen_test.xml')
tw_torus.setup_io()

Loading toroidal plasma mode
  filename = tCurr_mode.dat
  N        = 2
  # of pts = 200
  R0       = (1.0000E+00, -1.7087E-17)
  Mode pair sums -7.6050E-15 -1.5543E-15

Saving mesh: thincurr_mode.h5
#----------------------------------------------
Open FUSION Toolkit Initialized
Development branch:    main
Revision id:           1a4bc3c
Parallelization Info:
  Not compiled with MPI
  # of OpenMP threads =    4
Integer Precisions    =    4   8
Float Precisions      =    4   8  10
Complex Precisions    =    4   8
LA backend            = native
#----------------------------------------------


Creating thin-wall model
  No V(t) driver coils found
  No I(t) driver coils found
  Building holes

  Setup complete:
    # of points    =         6320
    # of edges     =        18960
    # of cells     =        12640
    # of holes     =            3
    # of closures  =            2
    # of Vcoils    =            0
    # of Icoils    =            0

Creating thin-wall model
  No V(t) driver co

In [2]:
tw_torus.compute_Lmat()
Lw = tw_torus.Lmat
print(Lw.shape)

Building element<->element self inductance matrix
  Time = 11s          


KeyboardInterrupt: 

In [9]:
print(Lw)

[[ 4.31728737e-02 -5.25175063e-04 -2.65307143e-03 ...  1.32983727e-06
  -1.78735289e-02 -1.95857014e-02]
 [-5.25175063e-04  5.07904292e-02 -1.03274169e-03 ...  1.85626872e-06
   2.13159381e-02  1.98079222e-02]
 [-2.65307143e-03 -1.03274169e-03  5.03946101e-02 ...  1.87062888e-06
   2.24651720e-02 -1.80428010e-02]
 ...
 [ 1.32983727e-06  1.85626872e-06  1.87062888e-06 ...  6.05431216e-02
  -3.13151597e-05  7.21002599e-04]
 [-1.78735289e-02  2.13159381e-02  2.24651720e-02 ... -3.13151597e-05
   1.66224988e+00  5.04700593e-03]
 [-1.95857014e-02  1.98079222e-02 -1.80428010e-02 ...  7.21002599e-04
   5.04700593e-03  6.74709927e+00]]


In [6]:
tw_mode.compute_Lmat()
Ld = tw_mode.Lmat

Building element<->element self inductance matrix
  Time = 31s          


In [8]:
print(Ld.shape)

(3162, 3162)


In [17]:
print(Ld)

[[ 6.87032210e-02 -4.43048484e-03 -1.60332977e-03 ... -2.47098149e-02
   1.37525148e-02  1.37525148e-02]
 [-4.43048484e-03  6.83811638e-02 -4.26899743e-03 ... -1.41374409e-02
   1.37527662e-02  1.37527662e-02]
 [-1.60332977e-03 -4.26899743e-03  6.78555133e-02 ... -7.67646173e-03
   1.37533082e-02  1.37533082e-02]
 ...
 [-2.47098149e-02 -1.41374409e-02 -7.67646173e-03 ...  6.54342016e+00
   1.46324962e-04  1.46324962e-04]
 [ 1.37525148e-02  1.37527662e-02  1.37533082e-02 ...  1.46324962e-04
   1.56870281e+00  2.74302987e-03]
 [ 1.37525148e-02  1.37527662e-02  1.37533082e-02 ...  1.46324962e-04
   2.74302987e-03  1.56870281e+00]]


In [36]:
La = np.array(Ld)
print(np.count_nonzero(np.isclose(La,1.37527662e-2)))

8


In [ ]:
Mwc = tw_torus.compute_Mcoil()
Lc = coils.compute_Lmat() 
    

In [ ]:
s = 
a = 
valen_model = VALENSystem(s,a,tw_torus,tw_mode)
eig_vals, eig_vecs = valen_model.eigenvalues()

Building element<->element self inductance matrix
  Time = 10s          
Building element<->element mutual inductance matrix
  Time = 32s          
Building element<->element self inductance matrix
  Time = 38s          
Building resistivity matrix
Building resistivity matrix


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 3162)